# L09 · Temporal Curriculum OPD

## Goal

**Estimated time:** 40 min · **Path:** full

- build a depth schedule
- distinguish F2B and B2F
- audit selected turns

### Current position: L08 → **L09** → L10

```text
Prompt/Data -> state source -> ... -> L09 -> ... -> fair evaluation
```

Alt text: The course map highlights L09 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L09"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L09', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

TCOD does not distill a full long trajectory from the start. F2B grows from early turns; B2F hands the ending to the student after a teacher/successful prefix.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

TCOD does not hand the student an entire long trajectory immediately. F2B grows from early turns and practices initial decisions. B2F begins at an easier suffix after a successful/teacher prefix.

Curriculum depth and pacing are distinct: depth is the number of currently included turns; pacing controls optimizer steps between depth increases. Sliced trajectories must preserve original observation/action boundaries and aligned loss masks.

### Production implementation: why this design

`curriculum_depth` derives depth from step and pacing with integer arithmetic. `temporal_curriculum_mask` returns only the intersection of F2B/B2F-selected turns and the response mask. Teacher-prefix generation for B2F is an explicit mini-backend approximation.

Production code: [`tcod.py`](../../src/opd_study/algorithms/tcod.py), [`advanced.py`](../../src/opd_study/training/advanced.py).

In [2]:
import inspect
from opd_study.algorithms.tcod import temporal_curriculum_mask, tcod_loss

objects_to_show = (temporal_curriculum_mask, tcod_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.tcod.temporal_curriculum_mask
def temporal_curriculum_mask(
    trajectories: TrajectoryBatch, *, depth: int, direction: str
) -> Tensor:
    """Select early F2B or late B2F turns from a correctly sourced trajectory.

    For B2F, callers must supply trajectories whose earlier history came from the
    teacher/successful prefix.  A mask alone cannot manufacture that state distribution.
    """

    if trajectories.turn_ids is None:
        raise ValueError("TCOD requires turn_ids")
    if depth < 1:
        raise ValueError("depth must be positive")
    turns = trajectories.turn_ids
    valid = trajectories.response_mask & torch.ge(turns, 0)
    if not valid.any().item():
        raise ValueError("TCOD found no response turns")
    if direction == "f2b":
        return valid & torch.lt(turns, depth)
    if direction == "b2f":
        maximum_turn = torch.where(valid, turns, torch.full_like(turns, -1)).amax(
            dim=1, keepdim=True
        )
        retur

### Alternatives and trade-offs

F2B is natural when early decisions dominate errors; B2F helps when long horizons make successful experience sparse. Random windows or difficulty curricula are alternatives, but calling them TCOD requires preserving direction and prefix conditions.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L09's output? Write one sentence, then run.

In [3]:
from opd_study.algorithms.tcod import curriculum_depth, temporal_curriculum_mask
from opd_study.data import CharacterTokenizer, collate_multiturn_text, generate_tiny_arithmetic

tokenizer = CharacterTokenizer(); rows = generate_tiny_arithmetic(train_rows=2, validation_rows=1, test_rows=1).train
batch = collate_multiturn_text([(row.prompt, tuple(row.response.splitlines())) for row in rows], tokenizer)
schedule = [curriculum_depth(step, start_depth=1, pacing_steps=2, maximum_depth=3)
            for step in range(6)]
print("depth schedule:", schedule)
for direction in ("f2b", "b2f"):
    mask = temporal_curriculum_mask(batch, depth=1, direction=direction)
    print(direction, "selected tokens:", int(mask.sum()))

depth schedule: [1, 1, 2, 2, 3, 3]
f2b selected tokens: 30
b2f selected tokens: 24


In [4]:
early = temporal_curriculum_mask(batch, depth=1, direction="f2b")
late = temporal_curriculum_mask(batch, depth=1, direction="b2f")
print("F2B learns beginnings; B2F needs a successful/teacher prefix before this suffix mask.")
print("overlap between one-turn windows:", int((early & late).sum()))

F2B learns beginnings; B2F needs a successful/teacher prefix before this suffix mask.
overlap between one-turn windows: 0


## Checks

In [5]:
assert schedule == [1, 1, 2, 2, 3, 3]
assert not (early & late).any()
assert batch.turn_ids is not None
print("check passed: pacing and temporal slices are explicit")

check passed: pacing and temporal slices are explicit


**Exercise (8 min):** tabulate F2B/B2F masks at depths 1, 2, and 3 for a three-turn batch, including the first step each turn appears.

<details><summary>Check</summary>F2B grows from low turn IDs, B2F from high IDs; distinguish pacing from depth.</details>

## My recurring mistakes

### M1 — Implementing B2F as a reversed mask

- Wrong: select a suffix after a failed student prefix.
- Why: a successful/teacher prefix is central to B2F.
- Fix: record prefix provenance and selected turns.
- Related check: `test_tcod_curriculum_and_directions`

### M2 — Equating depth with global step

- Wrong: add one turn every optimizer step.
- Why: pacing disappears.
- Fix: name start depth, pacing steps, and maximum depth.
- Related check: `test_tcod_curriculum_and_directions`

## 60-second summary

1. build a depth schedule
2. distinguish F2B and B2F
3. audit selected turns

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`tcod`](https://arxiv.org/abs/2604.24005v3) · `2604.24005v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`tcod_official`](https://github.com/kokolerk/TCOD) · `465eef4406ad0cff675b36bd46f37f28b1736ff9` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)